# Pipeline de Coleta SciELO/ArticleMeta - Base de Dados do Pipeline ABNT

Este notebook implementa a etapa de ingestao do sistema.

- consulta a API ArticleMeta/SciELO para localizar revistas brasileiras e identificadores de artigos
- baixa PDFs a partir do portal SciELO
- extrai texto com PyMuPDF (`fitz`)
- grava os CSVs base consumidos pelo notebook analitico: `artigos_completo.csv`, `artigos_validos.csv` e `artigos_limpos.csv`

Fluxo real: Configuracao -> Revistas brasileiras na ArticleMeta -> Identificadores de artigos -> Download SciELO PDF -> Extracao PyMuPDF -> CSVs base do pipeline.

Este notebook nao executa a avaliacao ABNT; ele prepara a base textual que sera analisada no notebook principal.


## Uso etico e limitacoes

**Uso etico:**
- este notebook deve respeitar os limites e termos de uso da infraestrutura SciELO e da API ArticleMeta
- a coleta e destinada a fins academicos e analiticos, com identificacao explicita da fonte dos dados
- o conteudo coletado nao deve ser redistribuido fora das condicoes de licenca de cada artigo

**Limitacoes metodologicas:**
- a amostra depende dos filtros adotados neste notebook, especialmente colecao, intervalo temporal e criterio de revista
- a extracao textual por PDF pode introduzir ruido de formatacao, hifenizacao e perda de estrutura
- os CSVs gerados sao uma base derivada de pesquisa e nao substituem os metadados e textos originais do SciELO
- a etapa de coleta nao realiza validacao ABNT, avaliacao de qualidade cientifica ou verificacao juridica de licenca por artigo


In [ ]:
%pip install requests pymupdf tqdm pandas --quiet

In [ ]:
import os
import time
import hashlib
import requests
import pandas as pd
import fitz  # PyMuPDF
from tqdm import tqdm

BASE_PATH  = os.path.join(os.getcwd(), "PLN_SCIELO")
DIR_PDFS   = os.path.join(BASE_PATH, "raw_pdfs")
DIR_TEXTOS = os.path.join(BASE_PATH, "textos")
for d in [DIR_PDFS, DIR_TEXTOS]:
    os.makedirs(d, exist_ok=True)

COLLECTION = "scl"
ANO_INICIO = "2020-01-01"
ANO_FIM    = "2023-12-31"
META_PDFS  = 3000
HEADERS    = {"User-Agent": "Mozilla/5.0 (compatible; research-bot/1.0)"}
API_BASE   = "https://articlemeta.scielo.org/api/v1"

print("Ambiente configurado.")
print(f"BASE_PATH: {BASE_PATH}")

In [ ]:
resp = requests.get(
    f"{API_BASE}/journal/",
    params={"collection": COLLECTION, "limit": 5, "offset": 0},
    headers=HEADERS,
    timeout=30,
)

resp.raise_for_status()
dados = resp.json()

print(f"Status HTTP: {resp.status_code}")
print(f"Registros retornados para amostra: {len(dados)}")
print("Campos disponíveis no primeiro registro:")
print(list(dados[0].keys())[:20])

## Etapa 1 - Buscar revistas brasileiras na API ArticleMeta

In [ ]:
def buscar_revistas_brasileiras_articlemeta(collection="scl"):
    revistas_brasileiras = []

    print("Buscando revistas brasileiras na ArticleMeta...")
    resp = requests.get(
        f"{API_BASE}/journal/",
        params={"collection": collection},
        headers=HEADERS,
        timeout=30,
    )
    resp.raise_for_status()

    dados = resp.json()
    print(f"Total de revistas na coleção: {len(dados)}")

    for revista in dados:
        v310 = revista.get("v310", [])
        pais = v310[0].get("_", "") if v310 else ""

        v350 = revista.get("v350", [])
        idiomas = [item.get("_", "") for item in v350]

        v930 = revista.get("v930", [])
        v100 = revista.get("v100", [])
        sigla = v930[0].get("_", "").lower() if v930 else ""
        titulo = v100[0].get("_", "") if v100 else ""

        condicao = (
            pais == "BR"
            and "pt" in idiomas
            and "revista brasileira" in titulo.lower()
            and sigla
            and revista.get("code", "")
        )
        if condicao:
            revistas_brasileiras.append(
                {
                    "issn": revista.get("code", ""),
                    "titulo": titulo,
                    "sigla": sigla,
                }
            )

    df_revistas = pd.DataFrame(revistas_brasileiras).drop_duplicates(subset=["issn"]).reset_index(drop=True)
    return df_revistas


buscar_revistas_rb = buscar_revistas_brasileiras_articlemeta

df_revistas_brasileiras = buscar_revistas_brasileiras_articlemeta(collection=COLLECTION)
print(f"Total de revistas encontradas: {len(df_revistas_brasileiras)}")
df_revistas_brasileiras.head(10)

## Etapa 2 - Coletar identificadores de artigos na ArticleMeta (2020-2023)

In [ ]:
def coletar_identificadores_artigos_articlemeta(df_revistas, ano_inicio, ano_fim):
    registros_artigos = []
    limit = 100

    for _, revista in tqdm(df_revistas.iterrows(), total=len(df_revistas), desc="Revistas"):
        offset = 0
        while True:
            resp = requests.get(
                f"{API_BASE}/article/identifiers/",
                params={
                    "collection": COLLECTION,
                    "issn": revista["issn"],
                    "limit": limit,
                    "offset": offset,
                    "from": ano_inicio,
                    "until": ano_fim,
                },
                headers=HEADERS,
                timeout=30,
            )

            if resp.status_code != 200:
                break

            dados = resp.json()
            objetos = dados.get("objects", []) if isinstance(dados, dict) else []

            for obj in objetos:
                registros_artigos.append(
                    {
                        "issn": revista["issn"],
                        "sigla": revista["sigla"],
                        "titulo_revista": revista["titulo"],
                        "pid": obj.get("code", ""),
                        "doi": obj.get("doi", ""),
                        "processing_date": obj.get("processing_date", ""),
                    }
                )

            if len(objetos) < limit:
                break

            offset += limit
            time.sleep(0.2)

    df_identificadores = pd.DataFrame(registros_artigos)
    if not df_identificadores.empty:
        df_identificadores = df_identificadores.drop_duplicates(subset=["pid"]).reset_index(drop=True)
    return df_identificadores


coletar_pids = coletar_identificadores_artigos_articlemeta

if len(df_revistas_brasileiras) == 0:
    raise RuntimeError("Nenhuma revista alvo encontrada. Revise os filtros da Etapa 1.")

df_identificadores_artigos = coletar_identificadores_artigos_articlemeta(
    df_revistas_brasileiras, ANO_INICIO, ANO_FIM
)
print(f"Total de artigos encontrados (PIDs únicos): {len(df_identificadores_artigos)}")
df_identificadores_artigos.head()

## Etapa 3 - Baixar PDFs do SciELO e extrair texto com PyMuPDF

In [ ]:
def _id_estavel(pid: str) -> str:
    # hash() do Python muda entre sessoes; sha1 mantem nome estavel de arquivo
    return hashlib.sha1(pid.encode("utf-8")).hexdigest()[:16]


def baixar_pdf_scielo_e_extrair_texto(row):
    pid, sigla = row["pid"], row["sigla"]
    nome_base = _id_estavel(pid)
    caminho_pdf = os.path.join(DIR_PDFS, f"{nome_base}.pdf")
    caminho_txt = os.path.join(DIR_TEXTOS, f"{nome_base}.txt")

    registro = row.copy()
    registro.update(
        {
            "status": None,
            "caminho_local": None,
            "caminho_texto": None,
            "texto_extraido": False,
            "n_caracteres": 0,
            "texto": "",
        }
    )

    if os.path.exists(caminho_pdf):
        registro["status"] = "ja_existia"
        registro["caminho_local"] = caminho_pdf
        if os.path.exists(caminho_txt):
            with open(caminho_txt, encoding="utf-8") as f:
                texto = f.read()
            registro.update(
                {
                    "caminho_texto": caminho_txt,
                    "texto_extraido": bool(texto),
                    "n_caracteres": len(texto),
                    "texto": texto,
                }
            )
        return registro

    pdf_url = f"https://www.scielo.br/j/{sigla}/a/{pid}/?format=pdf"

    try:
        resposta_pdf = requests.get(pdf_url, headers=HEADERS, timeout=20)
        e_pdf = "pdf" in resposta_pdf.headers.get("Content-Type", "").lower()
        if not (resposta_pdf.status_code == 200 and len(resposta_pdf.content) > 1000 and e_pdf):
            registro["status"] = "erro"
            return registro

        with fitz.open(stream=resposta_pdf.content, filetype="pdf") as doc:
            texto = "\n".join(pagina.get_text("text") for pagina in doc).strip()

        with open(caminho_pdf, "wb") as f:
            f.write(resposta_pdf.content)
        with open(caminho_txt, "w", encoding="utf-8") as f:
            f.write(texto)

        registro.update(
            {
                "status": "baixado",
                "caminho_local": caminho_pdf,
                "caminho_texto": caminho_txt,
                "texto_extraido": bool(texto),
                "n_caracteres": len(texto),
                "texto": texto,
            }
        )

    except Exception:
        registro["status"] = "erro"

    time.sleep(0.3)
    return registro


processar_pdf = baixar_pdf_scielo_e_extrair_texto

if df_identificadores_artigos.empty:
    raise RuntimeError("DataFrame de identificadores está vazio. Verifique a Etapa 2.")

registros_processados = []
limite_execucao = min(META_PDFS, len(df_identificadores_artigos))
for _, row in tqdm(
    df_identificadores_artigos.head(limite_execucao).iterrows(),
    total=limite_execucao,
    desc="PDFs"
):
    registros_processados.append(baixar_pdf_scielo_e_extrair_texto(row.to_dict()))

df_completo = pd.DataFrame(registros_processados)
df_validos = (
    df_completo[df_completo["status"].isin(["baixado", "ja_existia"])]
    .drop_duplicates(subset=["pid"])
    .reset_index(drop=True)
)

print(f"Total processado   : {len(df_completo)}")
print(f"  Baixados         : {(df_completo['status'] == 'baixado').sum()}")
print(f"  Ja existiam      : {(df_completo['status'] == 'ja_existia').sum()}")
print(f"  Erros            : {(df_completo['status'] == 'erro').sum()}")
print(f"PDFs validos unicos: {len(df_validos)}")

df_validos.head()

## Etapa 4 - Salvar CSVs base consumidos pelo notebook analitico

In [ ]:
# CSV 1: completo (tudo que foi processado)
df_artigos_completo = df_completo.copy()

# CSV 2: validos (baixados/existentes, sem duplicar PID)
df_artigos_validos = df_validos.copy()

# CSV 3: limpos (enxuto para PLN)
df_artigos_limpos = df_artigos_validos[["texto", "n_caracteres"]].reset_index(drop=True)

# Garante presenca de colunas esperadas no CSV de validos
colunas_esperadas = [
    "issn",
    "sigla",
    "titulo_revista",
    "pid",
    "doi",
    "processing_date",
    "status",
    "caminho_local",
    "caminho_texto",
    "texto_extraido",
    "n_caracteres",
    "texto",
]
for col in colunas_esperadas:
    if col not in df_artigos_validos.columns:
        df_artigos_validos[col] = ""

df_artigos_validos = df_artigos_validos[colunas_esperadas]

os.makedirs(BASE_PATH, exist_ok=True)

caminho_completo = os.path.join(BASE_PATH, "artigos_completo.csv")
caminho_validos  = os.path.join(BASE_PATH, "artigos_validos.csv")
caminho_limpos   = os.path.join(BASE_PATH, "artigos_limpos.csv")

df_artigos_completo.to_csv(caminho_completo, index=False, encoding="utf-8-sig")
df_artigos_validos.to_csv(caminho_validos, index=False, encoding="utf-8-sig")
df_artigos_limpos.to_csv(caminho_limpos, index=False, encoding="utf-8-sig")

print("Arquivos salvos com sucesso:")
print(f"- {caminho_completo}  ({len(df_artigos_completo)} linhas)")
print(f"- {caminho_validos}   ({len(df_artigos_validos)} linhas)")
print(f"- {caminho_limpos}    ({len(df_artigos_limpos)} linhas)")

df_artigos_validos.head(10)